### Base aggregated_by_filter - Indicadores agrupados por categoria e UF do vendedor

In [0]:

from pyspark.sql import functions as F

# Caminho da primeira base, com as previsões
PATH_PREDICTIONS = "/Volumes/workspace/olist_gold/ml_predictions/predictions"

# Caminho da segunda base, com os indicadores agregados
PATH_AGGREGATED = "/Volumes/workspace/olist_gold/ml_predictions/aggregated_by_filter"

# Ler as previsões salvas no Volume
df_predictions_spark = spark.read.parquet(PATH_PREDICTIONS)

# Agrupar e calcular os indicadores
df_aggregated = (
    df_predictions_spark
    .groupBy(
        "categoria",
        "seller_state",
        "prediction_type",
        "model_name",
        "model_version"
    )
    .agg(
        F.countDistinct("order_id").alias("total_pedidos"),

        F.avg("probability").alias("risco_medio_atraso"),

        F.sum("predicted_label").alias("pedidos_alto_risco"),

        (F.avg("predicted_label") * 100).alias("percentual_alto_risco"),

        F.avg("actual_label").alias("taxa_real_atraso_teste"),

        F.min("valor_produtos").alias("valor_produtos_min"),

        F.max("valor_produtos").alias("valor_produtos_max"),

        F.min("peso_total_g").alias("peso_total_g_min"),

        F.max("peso_total_g").alias("peso_total_g_max")
    )
)

# Salvar no Volume em formato Parquet
(
    df_aggregated.write
    .format("parquet")
    .mode("overwrite")
    .save(PATH_AGGREGATED)
)

print("Segunda base salva com sucesso!")


In [0]:

df_aggregated_check = spark.read.parquet(PATH_AGGREGATED)

print("Total de registros:", df_aggregated_check.count())
print("Total de colunas:", len(df_aggregated_check.columns))

display(
    df_aggregated_check
    .orderBy(F.desc("total_pedidos"))
    .limit(20)
)


### Base filter_options - Ela será responsável por armazenar as opções disponíveis para os filtros da aplicação, incluindo categorias, UFs e limites de preço e peso.

In [0]:

from pyspark.sql import functions as F

# Caminho da base principal de previsões
PATH_PREDICTIONS = "/Volumes/workspace/olist_gold/ml_predictions/predictions"

# Caminho da terceira base
PATH_FILTER_OPTIONS = "/Volumes/workspace/olist_gold/ml_predictions/filter_options"

# Ler as previsões
df_predictions_spark = spark.read.parquet(PATH_PREDICTIONS)

# Criar base com as opções e limites dos filtros
df_filter_options = (
    df_predictions_spark
    .groupBy("prediction_type", "model_name", "model_version")
    .agg(
        F.sort_array(
            F.collect_set("categoria")
        ).alias("categorias"),

        F.sort_array(
            F.collect_set("seller_state")
        ).alias("ufs_origem"),

        F.min("valor_produtos").alias("valor_produtos_min"),
        F.max("valor_produtos").alias("valor_produtos_max"),

        F.min("peso_total_g").alias("peso_total_g_min"),
        F.max("peso_total_g").alias("peso_total_g_max")
    )
)

# Salvar no Volume em formato Parquet
(
    df_filter_options.write
    .format("parquet")
    .mode("overwrite")
    .save(PATH_FILTER_OPTIONS)
)

print("Terceira base salva com sucesso!")


In [0]:

df_filter_options_check = spark.read.parquet(PATH_FILTER_OPTIONS)

print("Total de registros:", df_filter_options_check.count())
print("Total de colunas:", len(df_filter_options_check.columns))

display(df_filter_options_check)
